# Run reporting and teaching figures
Purpose: reproducible environment/provenance capture and figures for Notebook 00. Inputs: actual gate, mask, sensitivity and timing data. Outputs: JSON, PNG and PDF under explicit artifact paths. Definitions only on import. Expected runtime per figure <5 seconds. No training or checkpoints. Resume by rerunning the corresponding reporting cell using the saved raw result files.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, platform, subprocess, sys, uuid
import torch
import numpy as np
import matplotlib.pyplot as plt
from fmm_jepa.training import atomic_json, definition_hashes, config_hash

def environment_report(root,config):
    import psutil
    def command(args):
        completed=subprocess.run(args,capture_output=True,text=True)
        return completed.stdout.strip() if completed.returncode==0 else 'unavailable: '+completed.stderr.strip()
    return {'python':sys.version,'executable':sys.executable,'torch':torch.__version__,'cuda_build':torch.version.cuda,
            'cuda_available':torch.cuda.is_available(),'cuda_devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            'cuda_bf16_supported':bool(torch.cuda.is_bf16_supported()) if torch.cuda.is_available() else False,
            'platform':platform.platform(),'cpu_threads':torch.get_num_threads(),'available_ram_bytes':psutil.virtual_memory().available,
            'storage_mount':command(['findmnt','-T',str(root)]),'storage_persistence':'workspace is on persistent boot disk; instance deletion policy UNVERIFIED; no off-instance backup performed',
            'repository_commit':command(['git','-C',str(root),'rev-parse','HEAD']),
            'dtype_policy':'CPU float32 neural checks; CPU float64 identities/gradchecks; optional CUDA float32 smoke; mixed precision untested',
            'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),'tf32_matmul':torch.backends.cuda.matmul.allow_tf32,
            'time_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}

def prepare_run(root,config,resume_run_id=None):
    root=Path(root)
    run_id=resume_run_id or datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
    if Path(run_id).name!=run_id: raise ValueError('run_id must be a single path component')
    artifact=root/'artifacts'/'notebook_00'/run_id; checkpoint=root/'checkpoints'/run_id
    sources=definition_hashes(root)
    if resume_run_id:
        if not artifact.is_dir() or not (checkpoint/'manifest.json').is_file(): raise FileNotFoundError('Explicit resume run lacks artifacts/checkpoint manifest')
        old=json.loads((artifact/'resolved_config.json').read_text())
        if config_hash(config)!=config_hash(old):raise ValueError('Resume configuration mismatch')
    else:artifact.mkdir(parents=True,exist_ok=False)
    (artifact/'figures').mkdir(exist_ok=True)
    atomic_json(artifact/'resolved_config.json',config)
    return run_id,artifact,checkpoint,sources

def save_figure(figure,artifact,name):
    figure.savefig(Path(artifact)/'figures'/(name+'.png'),dpi=150,bbox_inches='tight')
    figure.savefig(Path(artifact)/'figures'/(name+'.pdf'),bbox_inches='tight')
    plt.show()

def style_figures():
    plt.rcParams.update({'figure.figsize':(9,4),'font.size':10,'axes.spines.top':False,'axes.spines.right':False,
                         'axes.labelcolor':'#262626','text.color':'#262626','axes.grid':False,'figure.dpi':110})

def architecture_figure():
    from matplotlib.patches import FancyBboxPatch
    fig,ax=plt.subplots(figsize=(11,3.7));ax.axis('off');ax.set(xlim=(0,11),ylim=(0,4))
    boxes=[(.2,1.5,1.5,.8,'Image patches'),(2.1,2.5,2.1,.8,'Visible patches\n+ original positions'),(2.1,.5,2.1,.8,'Complete grid'),
           (4.8,2.5,2,.8,'Context encoder\nFMM or ViT'),(4.8,.5,2,.8,'EMA teacher\nfrozen gradients'),(7.5,2.5,2.5,.8,'Transformer predictor\n+ target positions'),(7.5,.5,2.5,.8,'Select target latents\nfeature LayerNorm')]
    for x,y,w,h,label in boxes:
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.06',facecolor='#edf3f8',edgecolor='#245578'))
        ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=10)
    for a,b in [((1.7,1.9),(2.1,2.9)),((1.7,1.9),(2.1,.9)),((4.2,2.9),(4.8,2.9)),((4.2,.9),(4.8,.9)),((6.8,2.9),(7.5,2.9)),((6.8,.9),(7.5,.9))]:
        ax.annotate('',xy=b,xytext=a,arrowprops={'arrowstyle':'->','color':'#444444'})
    ax.annotate('EMA after optimizer',xy=(5.8,1.35),xytext=(5.8,2.18),ha='center',fontsize=9,arrowprops={'arrowstyle':'->','linestyle':'dashed'})
    ax.text(8.75,1.9,'SmoothL1 prediction loss',ha='center',weight='bold')
    ax.set_title('Only the encoder mixer changes; the predictor still uses attention',loc='left',pad=14)
    return fig

def mask_figure(grid,masks):
    rows,cols=grid
    fig,axes=plt.subplots(1,1+len(masks['target_indices']),figsize=(10,3),squeeze=False)
    context=np.zeros(rows*cols);context[masks['context_indices'][0][masks['context_valid'][0]].cpu().numpy()]=1
    axes[0,0].imshow(context.reshape(grid),vmin=0,vmax=1,cmap='Blues');axes[0,0].set_title('Visible context (blue)')
    for i,(indices,valid) in enumerate(zip(masks['target_indices'],masks['target_valid'])):
        target=np.zeros(rows*cols);target[indices[0][valid[0]].cpu().numpy()]=1
        axes[0,i+1].imshow(target.reshape(grid),vmin=0,vmax=1,cmap='Oranges');axes[0,i+1].set_title(f'Target block {i+1}')
    for ax in axes[0]:ax.set(xlabel='Original patch column',ylabel='Original patch row')
    fig.suptitle('Actual sampled mask; the union of targets is excluded from context')
    fig.tight_layout();return fig

def query_figure(grid=(8,8),query=27,dim=8):
    from fmm_jepa.geometry import pair_owner
    from fmm_jepa.operators import FMMMixer
    torch.manual_seed(197);count=grid[0]*grid[1]
    model=FMMMixer(dim);features=torch.randn(1,count,dim,requires_grad=True)
    idx=torch.arange(count)[None];valid=torch.ones_like(idx,dtype=torch.bool)
    output=model(features,idx,valid,grid)
    output[0,query].square().sum().backward()
    sensitivity=features.grad.norm(dim=-1)[0].reshape(grid).detach().numpy()
    ownership=np.array([pair_owner(query,source,grid) for source in range(count)]).reshape(grid)
    fig,axes=plt.subplots(1,2,figsize=(9,3.8))
    a=axes[0].imshow(ownership,cmap='cividis',vmin=0);fig.colorbar(a,ax=axes[0],label='0 = near; positive = far level',ticks=sorted(set(ownership.flatten())))
    b=axes[1].imshow(sensitivity,cmap='magma');fig.colorbar(b,ax=axes[1],label='Input-gradient norm')
    axes[0].set_title('Unique source-to-query owner');axes[1].set_title('Measured query sensitivity')
    for ax in axes:
        ax.scatter([query%grid[1]],[query//grid[1]],marker='x',s=95,color='cyan',linewidth=2)
        ax.set(xlabel='Source patch column',ylabel='Source patch row')
    fig.suptitle('Untrained controlled fixture; cyan × marks receiver patch')
    fig.tight_layout();return fig

def classical_figure(cases):
    fig,axes=plt.subplots(1,2,figsize=(9,3.5))
    for grid in sorted(set(tuple(c['grid']) for c in cases)):
        rows=sorted([c for c in cases if tuple(c['grid'])==grid],key=lambda c:c['order'])
        for ax,field,label in zip(axes,('absolute_l2_error','stabilized_relative_l2_error'),('Absolute L2 field error','Stabilized relative L2 error')):
            ax.semilogy([c['order'] for c in rows],[c[field] for c in rows],marker='o',label=f'{grid[0]} × {grid[1]}')
            ax.set(xlabel='Order p (nodes per axis)',ylabel=label,xticks=[2,4,6]);ax.grid(axis='y',alpha=.2);ax.legend()
    fig.suptitle('Classical regularized-kernel diagnostic; float64, fixed signed charges')
    fig.tight_layout();return fig

def training_figure(metrics):
    fig,axes=plt.subplots(1,3,figsize=(11,3.2));steps=[r['step'] for r in metrics]
    axes[0].plot(steps,[r['loss'] for r in metrics],marker='o',color='#245578');axes[0].set_ylabel('SmoothL1 loss')
    for key,label,style in [('context_std','Context','-'),('target_std','Teacher','--'),('prediction_std','Prediction',':')]:
        if key in metrics[0]: axes[1].plot(steps,[r[key] for r in metrics],linestyle=style,marker='.',label=label)
    axes[1].set_ylabel('Feature dispersion (mean std)');axes[1].legend(fontsize=8)
    for key,label,style in [('near_norm','Near','-'),('far_norm','Far','--')]:
        if key in metrics[0]:axes[2].plot(steps,[r[key] for r in metrics],linestyle=style,marker='.',label=label)
    axes[2].set_ylabel('Last context-block branch norm');axes[2].legend(fontsize=8)
    for ax in axes:ax.set_xlabel('Synthetic optimizer step');ax.grid(axis='y',alpha=.2)
    fig.suptitle('Synthetic smoke trace with changing masks and an EMA target; not a learning evaluation')
    fig.tight_layout();return fig

def benchmark_figure(rows):
    cpu=[r for r in rows if r['device']=='cpu' and r['mode']=='forward_backward']
    fig,axes=plt.subplots(1,3,figsize=(11,3.5))
    for ax,scope in zip(axes,('mixer','encoder_block','full_jepa')):
        for kind,color,marker in [('fmm','#245578','o'),('vit','#b96920','s')]:
            selected=sorted([r for r in cpu if r['scope']==scope and r['kind']==kind],key=lambda r:r['N_grid'])
            if not selected:continue
            ax.errorbar([r['N_grid'] for r in selected],[r['median_ms'] for r in selected],yerr=[r['mad_ms'] for r in selected],label=kind.upper(),color=color,marker=marker,capsize=3)
        ax.set(xlabel='Full-grid patch count',ylabel='Median forward + backward (ms)',title=scope.replace('_',' ').title(),yscale='log');ax.grid(axis='y',alpha=.2);ax.legend()
    fig.suptitle('Measured CPU timing; error bars are repeat MAD, not confidence intervals')
    fig.tight_layout();return fig


In [ ]:
def hierarchy_figure(grid=(7,9), leaf_size=2):
    from matplotlib.patches import Rectangle
    from fmm_jepa.geometry import build_geometry
    g=build_geometry(grid,leaf_size=leaf_size)
    fig,axes=plt.subplots(1,g.depth+1,figsize=(11,3.4),squeeze=False)
    extent_x=leaf_size*g.side/grid[1];extent_y=leaf_size*g.side/grid[0]
    for level,ax in enumerate(axes[0]):
        side=2**level;w=extent_x/side;h=extent_y/side
        ax.add_patch(Rectangle((0,0),extent_x,extent_y,facecolor='#f2f2f2',edgecolor='none'))
        ax.add_patch(Rectangle((0,0),1,1,facecolor='#dbe9f1',edgecolor='none'))
        for row in range(side):
            for col in range(side):ax.add_patch(Rectangle((col*w,row*h),w,h,fill=False,edgecolor='#777777',linewidth=.6))
        ax.scatter(g.coordinates[:,0],g.coordinates[:,1],s=5,color='#245578')
        ax.set(xlim=(0,extent_x),ylim=(extent_y,0),title=f'Level {level}: {4**level} cells',xlabel='Normalized x')
        ax.set_aspect('equal')
    axes[0,0].set_ylabel('Normalized y')
    fig.suptitle(f'{grid[0]} × {grid[1]} patches; blue = image extent, gray = allocated padding')
    fig.tight_layout();return fig
